In [1]:
# Homework (for me): Failure analysis. When a query fails, is it the entities, the structure, or both?
#  - Types of failure: Query executes, returns wrong answer. Query executes, returns nothing. Query fails altogether.

# Take queries with wrong entities, find the right entities, and re-run it. Does it work?

# If this holds, we may need to create a seperate step to retrieve the correct entities
# - Use special notation to take each keyword and perform a search

# We can take inspiration from the PGMR paper


In [5]:
import pandas as pd
import numpy as np 


#Defining failures

In [ ]:
# Does the query execute? - Yes/No
#If it executes, does it return an answer?
#Is the answer correct?

#How can we tell in the API whether the query was correct or not?

#Extract entity nouns from questions

In [24]:
data = pd.read_csv('mintaka_updated_v2.csv')


,Unnamed: 0,SAE Question,AAVE Question,SPARQL,Processed queries,Names
0,0,What is the seventh tallest mountain in North ...,Wha's da seventh tallest mountain in North Ame...,SELECT ?mountain ?mountainLabel ?height WHERE ...,[],NaN
1,1,Which actor was a star of Titanic and was born...,Which actor was da star of Titanic and was bor...,SELECT ?actor ?actorLabel WHERE { ?film rdfs:...,[{'actor': 'http://www.wikidata.org/entity/Q38...,Leonardo DiCaprio
2,2,Which actor starred in Vanilla Sky and was mar...,Which actor starred in Vanilla Sky an' was mar...,SELECT ?actor ?actorLabel WHERE { ?film rdfs:...,[{'actor': 'http://www.wikidata.org/entity/Q37...,Tom Cruise
3,3,In what year did the first book of the A Song ...,What year da first book of A Song of Ice and F...,SELECT ?book ?bookLabel ?year WHERE { ?book w...,[],NaN
4,4,Who is the youngest current U.S. governor?,Who da youngest current US governor?,SELECT ?person ?personLabel WHERE { ?person p...,[],NaN


In [31]:
import json
import requests
import pandas as pd
import time

USER_AGENT = "Colab-SAETOWikidata-SPARQL/1.0 (contact: saketsan8@gmail.com)"
WDQS_ENDPOINT = "https://query.wikidata.org/sparql"


def validate_query(q):
    if not isinstance(q, str): #If query is not a string
        return False
    q = q.strip()
    if not q:
        return False #If query is empty
    if not any(x in q.upper() for x in ["SELECT", "WHERE", "{", "}"]): #If no valid SPARQL
        return False
    return q.replace("\r", " ").replace("\n", " ")


def run_sparql(query, user_agent=USER_AGENT):
    q = validate_query(query)
    if not q:
        return {"valid": False, "error": "invalid or empty query", "bindings": []}

    try:
        r = requests.get(
            WDQS_ENDPOINT,
            params={"query": q},
            headers={
                "Accept": "application/sparql-results+json",
                "User-Agent": user_agent
            },
            timeout=60
        )
        r.raise_for_status()
        data = r.json()

    except Exception as e:
        return {"valid": False, "error": str(e), "query": q, "bindings": []}

    bindings = data.get("results", {}).get("bindings", [])
    parsed = [{k: v.get("value", "") for k, v in b.items()} for b in bindings]

    return {
        "valid": True,
        "bindings": parsed,
        "rows": len(parsed)
    }




# queries = []

# for index, row in eval.iterrows():
#     sparql = row["SPARQL"] if "SPARQL" in row and pd.notna(row["SPARQL"]) else ""
#     result = run_sparql(sparql)
#     queries.append(result)
#     time.sleep(1.5)
#     print(f"Row {index +1 } processed")


In [ ]:
import ast

evals = {}

def classify_sparql_response(data):
    """
    Classifies SPARQL query results into different error/success categories
    """
    
    # Handle string representation of data structures
    if isinstance(data, str):
        try:
            data = ast.literal_eval(data)
        except (ValueError, SyntaxError):
            return {
                "executed": False,
                "has_results": False,
                "is_correct": None,
                "error_type": "PARSE_ERROR",
                "message": "Could not parse response data",
                "raw_sample": data[:200]
            }
    
    # Check if it's a list (bindings directly)
    if isinstance(data, list):
        if len(data) == 0:
            return {
                "executed": True,
                "has_results": False,
                "is_correct": None,  # Can't determine correctness without ground truth
                "error_type": "EMPTY_RESULT",
                "message": "Query executed but returned no results",
                "bindings": []
            }
        return {
            "executed": True,
            "has_results": True,
            "is_correct": None,  # Needs manual verification or ground truth comparison
            "error_type": None,
            "message": "Query executed successfully with results",
            "bindings": data,
            "result_count": len(data)
        }
    
    # Check if it's a dict with SPARQL JSON structure
    if isinstance(data, dict):
        # Check for error indicators in the response
        if "error" in data or "message" in data:
            return {
                "executed": False,
                "has_results": False,
                "is_correct": False,
                "error_type": "QUERY_ERROR",
                "message": data.get("error", data.get("message", "Unknown error")),
                "raw_data": data
            }
        
        # Standard SPARQL JSON format
        if "results" in data and "bindings" in data["results"]:
            bindings = data["results"]["bindings"]
            if len(bindings) == 0:
                return {
                    "executed": True,
                    "has_results": False,
                    "is_correct": None,
                    "error_type": "EMPTY_RESULT",
                    "message": "Query executed but returned no results",
                    "bindings": []
                }
            return {
                "executed": True,
                "has_results": True,
                "is_correct": None,
                "error_type": None,
                "message": "Query executed successfully with results",
                "bindings": bindings,
                "result_count": len(bindings)
            }
        else:
            return {
                "executed": False,
                "has_results": False,
                "is_correct": False,
                "error_type": "INVALID_SPARQL_JSON",
                "message": "Response doesn't match SPARQL JSON format",
                "raw_data": data
            }
    
    # Unknown data type
    return {
        "executed": False,
        "has_results": False,
        "is_correct": None,
        "error_type": "INVALID_DATA_TYPE",
        "message": f"Unexpected data type: {type(data)}",
        "raw_sample": str(data)[:200]
    }

# Process all queries
for index, row in data.iterrows():
    query_result = row[' Processed queries']
    results = classify_sparql_response(query_result)
    evals[index] = results
    print(f'Index {index + 1}: {results["error_type"] or "SUCCESS"} - {results["message"]}')




# Generate summary statistics
print("\n=== EVALUATION SUMMARY ===")
total = len(evals)
executed = sum(1 for v in evals.values() if v["executed"])
has_results = sum(1 for v in evals.values() if v["has_results"])

print(f"Total queries: {total}")
print(f"Successfully executed: {executed} ({executed/total*100:.1f}%)")
print(f"Returned results: {has_results} ({has_results/total*100:.1f}%)")

# Error breakdown
error_types = {}
for v in evals.values():
    if v["error_type"]:
        error_types[v["error_type"]] = error_types.get(v["error_type"], 0) + 1

if error_types:
    print("\n=== ERROR BREAKDOWN ===")
    for error_type, count in sorted(error_types.items(), key=lambda x: x[1], reverse=True):
        print(f"{error_type}: {count} ({count/total*100:.1f}%)")

Index 1: EMPTY_RESULT - Query executed but returned no results
Index 2: SUCCESS - Query executed successfully with results
Index 3: SUCCESS - Query executed successfully with results
Index 4: EMPTY_RESULT - Query executed but returned no results
Index 5: EMPTY_RESULT - Query executed but returned no results
Index 6: SUCCESS - Query executed successfully with results
Index 7: EMPTY_RESULT - Query executed but returned no results
Index 8: EMPTY_RESULT - Query executed but returned no results
Index 9: EMPTY_RESULT - Query executed but returned no results
Index 10: EMPTY_RESULT - Query executed but returned no results
Index 11: SUCCESS - Query executed successfully with results
Index 12: EMPTY_RESULT - Query executed but returned no results
Index 13: EMPTY_RESULT - Query executed but returned no results
Index 14: EMPTY_RESULT - Query executed but returned no results
Index 15: EMPTY_RESULT - Query executed but returned no results
Index 16: SUCCESS - Query executed successfully with results


In [72]:
data['SPARQL'][0]

'SELECT ?mountain ?mountainLabel ?height WHERE {  ?mountain wdt:P31 wd:Q8502;            wdt:P2048 ?height;            wdt:P30 wd:Q49.  SERVICE wikibase:label { bd:serviceParam wikibase:language "en". }}ORDER BY DESC(?height)LIMIT 1 OFFSET 6'

In [71]:
data.head()

,Unnamed: 0,SAE Question,AAVE Question,SPARQL,Processed queries,Names
0,0,What is the seventh tallest mountain in North ...,Wha's da seventh tallest mountain in North Ame...,SELECT ?mountain ?mountainLabel ?height WHERE ...,[],NaN
1,1,Which actor was a star of Titanic and was born...,Which actor was da star of Titanic and was bor...,SELECT ?actor ?actorLabel WHERE { ?film rdfs:...,[{'actor': 'http://www.wikidata.org/entity/Q38...,Leonardo DiCaprio
2,2,Which actor starred in Vanilla Sky and was mar...,Which actor starred in Vanilla Sky an' was mar...,SELECT ?actor ?actorLabel WHERE { ?film rdfs:...,[{'actor': 'http://www.wikidata.org/entity/Q37...,Tom Cruise
3,3,In what year did the first book of the A Song ...,What year da first book of A Song of Ice and F...,SELECT ?book ?bookLabel ?year WHERE { ?book w...,[],NaN
4,4,Who is the youngest current U.S. governor?,Who da youngest current US governor?,SELECT ?person ?personLabel WHERE { ?person p...,[],NaN


#Search for the correct queries

#Rerunning with substitution

#Comparing outcomes